# Data

## Dataset Overview

In [1]:
library(ggplot2)
library(tidyr)
library(dplyr)
library(rstatix)
source("config.R")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘rstatix’


The following object is masked from ‘package:stats’:

    filter




In [ ]:
marketing.data <- read.csv(MARKETING_DATA)

str(marketing.data)

In [ ]:
marketing.data$converted <- as.logical(toupper(marketing.data$converted))

day.levels <- c("Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday")
marketing.data$most.ads.day <- factor(marketing.data$most.ads.day, levels = day.levels, ordered = TRUE)

marketing.data$test.group <- factor(marketing.data$test.group, levels = c("psa", "ad"))

summary(marketing.data)

In [ ]:
missing.idx <- which(!complete.cases(marketing.data))
length(missing.idx)

# Exploratory Data Analysis

In [ ]:
marketing.data %>%
  group_by(test.group) %>%
  summarise(converted_rate = mean(converted), .groups = "drop") %>%
  ggplot(aes(x = test.group, y = converted_rate, fill = test.group)) +
  geom_bar(stat = "identity") +
  geom_text(aes(label = scales::percent(converted_rate, accuracy = 0.01)),
            vjust = -0.5) +
  labs(title = "Conversion Rate by Test Group",
       x = "", y = "Conversion Rate") +
  theme(legend.position = "none")


In [ ]:
marketing.data %>%
  group_by(test.group, most.ads.hour) %>%
  summarise(converted_rate = mean(converted), .groups = "drop") %>%
  ggplot(aes(x = most.ads.hour, y = converted_rate, fill = test.group)) +
  geom_bar(stat = "identity") +
  facet_wrap(~ test.group) +
  labs(title = "Conversion Rate by Hour and Test Group", x = "Hour", y = "Conversion Rate") +
  theme(legend.position = "none")

In [ ]:
marketing.data %>%
  group_by(test.group, most.ads.day) %>%
  summarise(converted_rate = mean(converted), .groups = "drop") %>%
  ggplot(aes(x = most.ads.day, y = converted_rate, fill = test.group)) +
  geom_bar(stat = "identity") +
  facet_wrap(~ test.group) +
  scale_x_discrete(labels = 0:6) +
  labs(title = "Conversion Rate by Day and Test Group", x = "Day (0=Mon, 6=Sun)", y = "Conversion Rate") +
  theme(legend.position = "none")

In [ ]:
marketing.data %>%
  filter(test.group == "ad") %>%
  group_by(most.ads.day, most.ads.hour) %>%
  summarise(converted_rate = mean(converted), .groups = "drop") %>%
  ggplot(aes(x = most.ads.hour, y = most.ads.day, fill = converted_rate)) +
  geom_tile() +
  scale_fill_gradient(low = "white", high = "steelblue") +
  scale_x_continuous(breaks = seq(0, 23, by = 2)) +
  scale_y_discrete(labels = 0:6) +
  labs(title = "Conversion Rate Heatmap (ad group)", x = "Hour", y = "Day (0=Mon, 6=Sun)")


In [ ]:
marketing.data %>%
  filter(test.group == "ad") %>%
  group_by(most.ads.day, most.ads.hour) %>%
  summarise(
    n = n(),
    .groups = "drop"
  ) %>%
  ggplot(aes(x = most.ads.hour, y = most.ads.day, fill = n)) +
  geom_tile() +
  scale_fill_gradient(low = "white", high = "steelblue") +
  scale_x_continuous(breaks = seq(0, 23, by = 2)) +
  labs(title = "Sample Size Heatmap (ad group)", 
       x = "Hour", 
       y = "Day")

In [ ]:
Q1 <- quantile(marketing.data$total.ads, 0.25)
Q3 <- quantile(marketing.data$total.ads, 0.75)
IQR_val <- Q3 - Q1

lower_bound <- Q1 - 3 * IQR_val
upper_bound <- Q3 + 3 * IQR_val

outliers <- marketing.data[marketing.data$total.ads > upper_bound, ]

cat("Lower bound:", lower_bound, "\n")
cat("Upper bound:", upper_bound, "\n")
cat("Number of outliers:", nrow(outliers), "\n")
cat("Percentage:", round(nrow(outliers) / nrow(marketing.data) * 100, 2), "%\n")

ggplot(marketing.data, aes(x = test.group, y = total.ads, fill = test.group)) +
  geom_boxplot(alpha = 0.7) +
  geom_hline(yintercept = upper_bound, color = "red", linetype = "dashed") +
  scale_y_log10() +
  labs(title = "Outlier Threshold by Test Group (Log Scale)",
       x = "Test Group", y = "Total Ads (log)") +
  theme(legend.position = "none")


In [ ]:
marketing.data %>%
  mutate(is_outlier = total.ads > 96) %>%
  group_by(is_outlier, test.group) %>%
  summarise(converted_rate = mean(converted), .groups = "drop")%>%
  knitr::kable()


# Frequentist A/B Testing

## Randomization Check

In [ ]:
marketing.data %>%
  group_by(test.group) %>%
  summarise(
    mean = mean(total.ads),
    median = median(total.ads),
    variance = var(total.ads)
  )%>%
  knitr::kable()

In [ ]:
result_total_ads <- wilcox.test(total.ads ~ test.group, data = marketing.data)

n_ad  <- as.numeric(sum(marketing.data$test.group == "ad"))
n_psa <- as.numeric(sum(marketing.data$test.group == "psa"))

W <- as.numeric(result_total_ads$statistic)
rank_biserial_r <- 1 - (2 * W) / (n_ad * n_psa)

mw_results <- data.frame(
  metric          = "total.ads",
  W               = W,
  p_value         = round(result_total_ads$p.value, 4),
  rank_biserial_r = round(rank_biserial_r, 4)
)

print(mw_results, row.names = FALSE)

## Conversion Rate (converted)

In [ ]:
table.converted <- table(test.group = marketing.data$test.group, 
                         converted = marketing.data$converted)
prop.test(table.converted[, c("TRUE", "FALSE")])

## Subgroup Analysis

### Conversion Rate by most.ads.day (ad group only)

In [ ]:
chisq_day <- marketing.data %>%
  filter(test.group == "ad") %>%
  {table(.$most.ads.day, .$converted)} %>%
  chisq.test()

chisq_day

In [ ]:
marketing.data %>%
  filter(test.group == "ad") %>%
  group_by(most.ads.day) %>%
  summarise(
    n = n(),
    conversion_rate = mean(converted)
  ) %>%
  arrange(desc(conversion_rate)) %>%
  knitr::kable()

In [ ]:
chisq_day$residuals %>%
  knitr::kable()

#### Pairwise Comparison of Conversion Rate by most.ads.day (Bonferroni Correction)

In [ ]:
pairwise.prop.test(
  x = c(2778, 2270, 1963, 1711, 1995, 1679, 2027),
  n = c(83571, 74572, 77418, 79077, 88805, 78802, 82332),
  p.adjust.method = "bonferroni"
)

### Conversion Rate by most.ads.hour (ad group only)

In [ ]:
chisq_hour <- marketing.data %>%
  filter(test.group == "ad") %>%
  {table(.$most.ads.hour, .$converted)} %>%
  chisq.test()

chisq_hour

In [ ]:
marketing.data %>%
  filter(test.group == "ad") %>%
  group_by(most.ads.hour) %>%
  summarise(
    n = n(),
    conversion_rate = sum(converted) / n()
  ) %>%
  arrange(desc(conversion_rate)) %>%
  knitr::kable()

#### Pairwise Comparison of Conversion Rate by most.ads.hour (Bonferroni Correction)

In [ ]:
hour_summary <- marketing.data %>%
  filter(test.group == "ad") %>%
  group_by(most.ads.hour) %>%
  summarise(
    n = n(),
    converted = sum(converted),
    conversion_rate = sum(converted) / n()
  ) %>%
  slice_max(conversion_rate, n = 10) %>%
  arrange(most.ads.hour)

result <- pairwise.prop.test(
  x = hour_summary$converted,
  n = hour_summary$n,
  p.adjust.method = "bonferroni"
)

hour_labels <- as.character(hour_summary$most.ads.hour)

rownames(result$p.value) <- hour_labels[-1]
colnames(result$p.value) <- hour_labels[-length(hour_labels)]

result

# Bayesian A/B Testing